# Notebook 09 — Neural Depth Estimation

**Vision & 3D Mapping Workshop** | Block 3: Depth & 3D Reconstruction

---

## Why This Matters

A single camera cannot recover absolute depth from one image — the problem is
fundamentally **ill-posed** because infinitely many 3-D scenes project to the
same 2-D image. Yet neural networks have learned remarkably strong geometric
priors from massive training data, enabling **monocular depth estimation** that
rivals stereo in many scenarios.

This notebook covers the mathematical foundations — from the ambiguity of
monocular depth, through the self-attention mechanism that powers modern
architectures, to the metrics and loss functions used to train and evaluate
depth networks.

### The Evolution of Monocular Depth

The field has undergone three paradigm shifts (survey: Birkl et al., TechRxiv 2025;
Yang & Zhan, arXiv 2507.11540):

| Era | Approach | Key Papers | Resolution |
|-----|----------|------------|-----------|
| **2014–2018** | CNN encoder-decoder | Eigen et al. (NeurIPS 2014), DORN (CVPR 2018) | Single-domain, metric |
| **2019–2022** | Multi-dataset + ViT | MiDaS (TPAMI 2022), DPT (ICCV 2021) | Cross-domain, relative |
| **2023–2024** | Foundation models | Depth Anything v1/v2, Marigold, Metric3D v2 | Zero-shot, metric or relative |
| **2025–2026** | Unified mono/multi-view | DA v3 (ICLR 2026), FoundationSLAM (AAAI 2026) | Multi-frame, metric |

### What You'll Learn

1. **Why monocular depth is ill-posed** — the projection ambiguity
2. **Scale/shift ambiguity** — affine-invariant depth and how to align it
3. **DPT architecture** — ViT backbone, reassemble, fusion, depth head
4. **Self-attention math** — queries, keys, values, scaled dot-product, multi-head
5. **Depth Anything v2/v3** — DINOv2 backbone + DPT head
6. **DUSt3R / MASt3R** — predicting 3-D pointmaps directly
7. **Depth metrics** — abs_rel, sq_rel, RMSE, $\delta_1$, $\delta_2$, $\delta_3$
8. **Loss functions** — scale-invariant loss, gradient matching loss
9. **Exercises** — synthesize depth, add noise, compute metrics, align predictions

### Prerequisites

| Concept | Where |
|:---|:---|
| Pinhole camera, projection | Notebook 03 |
| Stereo depth, $Z = fB/d$ | Notebook 08 |
| Linear algebra, least-squares | Notebook 02 |

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
from scipy.ndimage import gaussian_filter, sobel

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)

%matplotlib inline
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "font.size": 12,
    "image.cmap": "turbo",
    "axes.grid": False,
})

---
## 1. Why Monocular Depth Is Ill-Posed

### 1.1 The Projection Ambiguity

A pinhole camera maps a 3-D point $\mathbf{P} = (X, Y, Z)^T$ to a 2-D pixel:

$$
\mathbf{x} = \pi(\mathbf{P}) = \frac{1}{Z} \begin{bmatrix} fX + c_x Z \\ fY + c_y Z \end{bmatrix}
$$

This projection is a **many-to-one** mapping. Every point on the ray

$$
\mathbf{P}(\lambda) = \lambda \, K^{-1} \begin{bmatrix} u \\ v \\ 1 \end{bmatrix}, \quad \lambda > 0
$$

projects to the same pixel $(u, v)$. Without additional constraints, there are
**infinitely many** 3-D scenes consistent with a single image.

### 1.2 What Makes It Solvable in Practice?

Neural networks exploit **learned priors**:

- **Perspective cues**: lines converging towards vanishing points
- **Texture gradients**: denser texture → farther away
- **Occlusion ordering**: closer objects occlude farther ones
- **Semantic knowledge**: "cars are ~4 m long, people are ~1.7 m tall"
- **Atmospheric perspective**: haze increases with distance

These cues let networks predict **relative** (ordinal or affine-invariant) depth
with impressive accuracy, but **absolute metric scale** remains ambiguous
without external information.

In [ ]:
def demonstrate_projection_ambiguity():
    """Show that multiple 3D scenes project to the same image."""
    f_cam = 500.0
    
    scales = [1.0, 2.0, 3.0]
    base_points = np.array([
        [-0.5, -0.3, 2.0],
        [ 0.5, -0.3, 2.0],
        [ 0.5,  0.3, 2.0],
        [-0.5,  0.3, 2.0],
    ])
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    colors = ['blue', 'red', 'green']
    for s, c in zip(scales, colors):
        pts_3d = base_points * s
        
        poly = plt.Polygon(pts_3d[:, [0, 2]], alpha=0.3, color=c,
                           label=f'Scale = {s}×  (Z = {pts_3d[0, 2]:.1f} m)')
        ax1.add_patch(poly)
        for pt in pts_3d:
            ax1.plot([0, pt[0]], [0, pt[2]], '--', color=c, alpha=0.3)
    
    ax1.plot(0, 0, 'ko', markersize=8, label='Camera')
    ax1.set_xlabel('X (m)')
    ax1.set_ylabel('Z (m)')
    ax1.set_title('Bird\'s Eye View: Multiple Scenes')
    ax1.legend(fontsize=9)
    ax1.set_xlim(-4, 4)
    ax1.set_ylim(-0.5, 8)
    ax1.set_aspect('equal')
    ax1.grid(True, alpha=0.3)
    
    for s, c in zip(scales, colors):
        pts_3d = base_points * s
        pts_2d = f_cam * pts_3d[:, :2] / pts_3d[:, 2:3]
        poly = plt.Polygon(pts_2d, alpha=0.3 + 0.15 * s, color=c, fill=False,
                           linewidth=3 - 0.5 * s)
        ax2.add_patch(poly)
    
    ax2.set_xlim(-300, 300)
    ax2.set_ylim(-200, 200)
    ax2.set_xlabel('u (pixels)')
    ax2.set_ylabel('v (pixels)')
    ax2.set_title('Image Plane: All Scenes Look Identical')
    ax2.set_aspect('equal')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

demonstrate_projection_ambiguity()

---
## 2. Scale/Shift Ambiguity

### 2.1 Affine-Invariant Depth

A monocular depth network outputs a **relative** depth map $d_{\text{net}}$ that
is related to the true metric depth $d_{\text{metric}}$ by an unknown affine
transformation:

$$
d_{\text{metric}} = \alpha \cdot d_{\text{net}} + \beta
$$

where:

- $\alpha > 0$ is the unknown **scale** factor
- $\beta$ is the unknown **shift** (offset)

### 2.2 Recovering Scale and Shift via Least-Squares

Given ground-truth depth $d_i^{\text{gt}}$ at $N$ valid pixels, we solve:

$$
\min_{\alpha, \beta} \sum_{i=1}^{N} \bigl(\alpha \, d_i^{\text{net}} + \beta - d_i^{\text{gt}}\bigr)^2
$$

This is a standard linear regression. In matrix form:

$$
\underbrace{\begin{bmatrix} d_1^{\text{net}} & 1 \\ d_2^{\text{net}} & 1 \\ \vdots & \vdots \\ d_N^{\text{net}} & 1 \end{bmatrix}}_{A}
\begin{bmatrix} \alpha \\ \beta \end{bmatrix}
= \underbrace{\begin{bmatrix} d_1^{\text{gt}} \\ d_2^{\text{gt}} \\ \vdots \\ d_N^{\text{gt}} \end{bmatrix}}_{\mathbf{b}}
$$

The normal equations give the closed-form solution:

$$
\begin{bmatrix} \alpha \\ \beta \end{bmatrix}
= (A^T A)^{-1} A^T \mathbf{b}
= \begin{bmatrix}
\sum d_i^2 & \sum d_i \\
\sum d_i & N
\end{bmatrix}^{-1}
\begin{bmatrix}
\sum d_i \, g_i \\
\sum g_i
\end{bmatrix}
$$

### 2.3 Median Scaling (Scale-Only)

When shift is negligible (e.g., outdoor scenes), a simpler approach:

$$
\alpha = \text{median}\left(\frac{d_i^{\text{gt}}}{d_i^{\text{net}}}\right)
$$

The median is robust to outliers (unlike the mean). Median scaling is the standard alignment method for benchmarking monocular depth in autonomous driving (KITTI, nuScenes), where the ground plane provides a reliable depth reference.

In [ ]:
def generate_synthetic_depth_gt(H=240, W=320):
    """Generate a synthetic ground-truth depth map."""
    y, x = np.mgrid[0:H, 0:W].astype(np.float32)
    depth = 2.0 + 8.0 * (y / H)
    
    depth[60:140, 50:130] = 2.5
    depth[80:180, 180:280] = 5.0
    depth[30:80, 220:300] = 3.5
    depth[160:220, 80:160] = 6.0
    
    depth = gaussian_filter(depth, sigma=2.0)
    return depth


def simulate_network_prediction(depth_gt, scale=0.4, shift=1.5, noise_std=0.3):
    """Simulate an affine-ambiguous network prediction."""
    d_net = (depth_gt - shift) / scale
    d_net += np.random.randn(*d_net.shape) * noise_std
    return d_net


def align_depth_least_squares(d_pred, d_gt, mask=None):
    """Solve for optimal α, β via least-squares."""
    if mask is None:
        mask = d_gt > 0
    d = d_pred[mask].ravel().astype(np.float64)
    g = d_gt[mask].ravel().astype(np.float64)
    
    A = np.stack([d, np.ones_like(d)], axis=1)
    result, _, _, _ = np.linalg.lstsq(A, g, rcond=None)
    alpha, beta = float(result[0]), float(result[1])
    
    aligned = alpha * d_pred + beta
    return alpha, beta, aligned


depth_gt = generate_synthetic_depth_gt()
d_net = simulate_network_prediction(depth_gt, scale=0.4, shift=1.5, noise_std=0.3)

alpha, beta, d_aligned = align_depth_least_squares(d_net, depth_gt)

print(f"True affine: d_metric = 0.4 · d_net + 1.5")
print(f"Recovered:   d_metric = {alpha:.4f} · d_net + {beta:.4f}")

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
im0 = axes[0].imshow(depth_gt, cmap='turbo')
axes[0].set_title('Ground Truth Depth')
plt.colorbar(im0, ax=axes[0], shrink=0.8)

im1 = axes[1].imshow(d_net, cmap='turbo')
axes[1].set_title('Network Output (ambiguous)')
plt.colorbar(im1, ax=axes[1], shrink=0.8)

im2 = axes[2].imshow(d_aligned, cmap='turbo')
axes[2].set_title(f'Aligned (α={alpha:.3f}, β={beta:.3f})')
plt.colorbar(im2, ax=axes[2], shrink=0.8)

error = np.abs(d_aligned - depth_gt)
im3 = axes[3].imshow(error, cmap='hot', vmin=0, vmax=2)
axes[3].set_title(f'|Error| (mean={error.mean():.3f} m)')
plt.colorbar(im3, ax=axes[3], shrink=0.8)

for ax in axes:
    ax.axis('off')
plt.suptitle('Scale/Shift Alignment via Least-Squares', fontsize=14)
plt.tight_layout()
plt.show()

# --- 3D Point Cloud from Back-Projected Depth Maps ---
from mpl_toolkits.mplot3d import Axes3D

H_d, W_d = depth_gt.shape
fx_cam, fy_cam = 500.0, 500.0
cx_cam, cy_cam = W_d / 2.0, H_d / 2.0

v_coords, u_coords = np.mgrid[0:H_d, 0:W_d]

Z_gt = depth_gt
X_gt = (u_coords - cx_cam) * Z_gt / fx_cam
Y_gt = (v_coords - cy_cam) * Z_gt / fy_cam

Z_al = d_aligned
X_al = (u_coords - cx_cam) * Z_al / fx_cam
Y_al = (v_coords - cy_cam) * Z_al / fy_cam

step = 4
fig = plt.figure(figsize=(16, 6))

ax1 = fig.add_subplot(121, projection='3d')
sc1 = ax1.scatter(
    X_gt[::step, ::step].ravel(),
    Y_gt[::step, ::step].ravel(),
    Z_gt[::step, ::step].ravel(),
    c=Z_gt[::step, ::step].ravel(),
    cmap='viridis', s=1, alpha=0.6
)
ax1.set_xlabel('X (m)')
ax1.set_ylabel('Y (m)')
ax1.set_zlabel('Z — Depth (m)')
ax1.set_title('Ground Truth Depth → 3D Point Cloud')
ax1.view_init(elev=30, azim=-45)
fig.colorbar(sc1, ax=ax1, shrink=0.5, label='Depth (m)')

ax2 = fig.add_subplot(122, projection='3d')
sc2 = ax2.scatter(
    X_al[::step, ::step].ravel(),
    Y_al[::step, ::step].ravel(),
    Z_al[::step, ::step].ravel(),
    c=Z_al[::step, ::step].ravel(),
    cmap='viridis', s=1, alpha=0.6
)
ax2.set_xlabel('X (m)')
ax2.set_ylabel('Y (m)')
ax2.set_zlabel('Z — Depth (m)')
ax2.set_title(f'Aligned Prediction → 3D Point Cloud\n(α={alpha:.3f}, β={beta:.3f})')
ax2.view_init(elev=30, azim=-45)
fig.colorbar(sc2, ax=ax2, shrink=0.5, label='Depth (m)')

plt.suptitle('3D Back-Projection: Depth Maps → Point Clouds', fontsize=14)
plt.tight_layout()
plt.show()

print("Back-projection: X = (u - cx) · Z / fx,  Y = (v - cy) · Z / fy")
print(f"Camera intrinsics: fx=fy={fx_cam}, cx={cx_cam}, cy={cy_cam}")
print(f"Point cloud size: {X_gt[::step, ::step].size} points (subsampled {step}×)")

---
## 3. DPT Architecture

The **Dense Prediction Transformer (DPT)** repurposes a Vision Transformer (ViT)
for pixel-level tasks like depth estimation.

### 3.1 Pipeline Overview

$$
\text{Image} \xrightarrow{\text{patch embed}}
\text{Tokens} \xrightarrow{\text{Transformer}}
\text{Multi-scale tokens} \xrightarrow{\text{Reassemble}}
\text{Feature maps} \xrightarrow{\text{Fusion}}
\text{Depth map}
$$

### 3.2 Patch Embedding

An image $I \in \mathbb{R}^{H \times W \times 3}$ is split into non-overlapping
patches of size $p \times p$ (typically $p = 16$). Each patch is linearly
embedded into a $D$-dimensional token:

$$
\mathbf{z}_0^{(i)} = \text{flatten}(\text{patch}_i) \cdot W_E + \mathbf{b}_E, \qquad
W_E \in \mathbb{R}^{(p^2 \cdot 3) \times D}
$$

Number of tokens: $N = \frac{H}{p} \times \frac{W}{p}$

A learnable **[CLS]** token and **positional embeddings** are prepended/added:

$$
\mathbf{Z}_0 = \bigl[\mathbf{z}_{\text{cls}}; \, \mathbf{z}_0^{(1)}; \, \ldots; \, \mathbf{z}_0^{(N)}\bigr] + \mathbf{E}_{\text{pos}}
$$

**Positional encoding variants.** The original Transformer (Vaswani et al., 2017) uses
fixed **sinusoidal** positional encodings. For token position $\text{pos}$ and dimension
index $i \in \{0, 1, \ldots, D/2 - 1\}$:

$$
\text{PE}(\text{pos}, 2i) = \sin\!\left(\frac{\text{pos}}{10000^{2i/D}}\right), \qquad
\text{PE}(\text{pos}, 2i+1) = \cos\!\left(\frac{\text{pos}}{10000^{2i/D}}\right)
$$

Each dimension oscillates at a distinct frequency $\omega_i = 1 / 10000^{2i/D}$, forming a
geometric series from high frequency ($i = 0$, period $2\pi$) to low frequency
($i = D/2 - 1$, period $2\pi \cdot 10000$).

**Why sinusoids encode relative position.** For any fixed offset $k$,
$\text{PE}(\text{pos} + k)$ is a linear function of $\text{PE}(\text{pos})$:

$$
\begin{pmatrix} \sin(\omega_i(\text{pos}+k)) \\ \cos(\omega_i(\text{pos}+k)) \end{pmatrix}
= \begin{pmatrix} \cos(\omega_i k) & \sin(\omega_i k) \\ -\sin(\omega_i k) & \cos(\omega_i k) \end{pmatrix}
\begin{pmatrix} \sin(\omega_i \, \text{pos}) \\ \cos(\omega_i \, \text{pos}) \end{pmatrix}
$$

This rotation-matrix structure means an attention head can learn to attend to
"the token $k$ positions away" by learning a linear projection of the positional
dimensions — the relative offset $k$ determines a fixed rotation matrix that the
dot-product attention can detect.

**ViT and DPT** use **learnable** positional embeddings $\mathbf{E}_{\text{pos}} \in
\mathbb{R}^{(N+1) \times D}$ (optimised during training) rather than fixed sinusoids.
For 2-D images the embedding is either a flat 1-D index over the $N$ patches
or a factored 2-D grid (separate row and column embeddings, concatenated or summed).
Empirically, learnable embeddings match or slightly exceed sinusoidal performance
for ViT-scale models (Dosovitskiy et al., 2021).

### 3.3 Transformer Encoder

Each of $L$ transformer layers applies multi-head self-attention (MHSA) and a
feed-forward network (FFN) with residual connections and layer normalisation:

$$
\mathbf{Z}'_l = \mathbf{Z}_{l-1} + \text{MHSA}\bigl(\text{LN}(\mathbf{Z}_{l-1})\bigr)
$$
$$
\mathbf{Z}_l = \mathbf{Z}'_l + \text{FFN}\bigl(\text{LN}(\mathbf{Z}'_l)\bigr)
$$

where $\text{FFN}(\mathbf{x}) = W_2 \, \text{GELU}(W_1 \mathbf{x} + \mathbf{b}_1) + \mathbf{b}_2$.

### 3.4 Reassemble + Fusion

DPT extracts tokens from **four** intermediate layers (e.g., layers 3, 6, 9, 12)
to capture multi-scale features:

- **Reassemble**: reshape tokens from $(N, D)$ back to spatial maps of size
  $\frac{H}{p} \times \frac{W}{p} \times D$, then apply a $1 \times 1$ convolution
  to project to the fusion dimension, and resize to the target resolution.

- **Fusion**: progressively upsample and merge feature maps using residual
  convolution blocks:

$$
F_l = \text{ResConv}\bigl(\text{Upsample}(F_{l+1}) + R_l\bigr)
$$

  from coarsest ($l = 4$) to finest ($l = 1$), where $R_l$ is the reassembled
  feature at level $l$.

### 3.5 Depth Head

A final convolutional layer maps the fused features to per-pixel depth:

$$
\hat{d} = \text{Conv}_{1 \times 1}(F_1) \in \mathbb{R}^{H \times W}
$$

In [ ]:
def visualize_dpt_architecture():
    """Create a schematic of the DPT architecture."""
    fig, ax = plt.subplots(figsize=(16, 6))
    ax.set_xlim(0, 16)
    ax.set_ylim(0, 6)
    ax.axis('off')
    
    blocks = [
        (0.5, 2.5, 1.5, 2.0, 'Image\n(H×W×3)', '#FFB3BA'),
        (2.5, 2.5, 1.5, 2.0, 'Patch\nEmbed\n(N×D)', '#BAFFC9'),
        (4.5, 2.5, 2.0, 2.0, 'ViT\nTransformer\n(L layers)', '#BAE1FF'),
        (7.0, 2.5, 1.8, 2.0, 'Reassemble\n4 scales', '#FFFFBA'),
        (9.3, 2.5, 1.8, 2.0, 'Fusion\n(upsample\n+ merge)', '#E8BAFF'),
        (11.6, 2.5, 1.5, 2.0, 'Depth\nHead\n(1×1 conv)', '#FFDFBA'),
        (13.6, 2.5, 1.5, 2.0, 'Depth\nMap\n(H×W)', '#C9FFE5'),
    ]
    
    for x, y, w, h, text, color in blocks:
        rect = plt.Rectangle((x, y), w, h, linewidth=2, edgecolor='black',
                             facecolor=color, zorder=2)
        ax.add_patch(rect)
        ax.text(x + w/2, y + h/2, text, ha='center', va='center',
                fontsize=9, fontweight='bold', zorder=3)
    
    arrow_starts = [2.0, 4.0, 6.5, 8.8, 11.1, 13.1]
    arrow_ends   = [2.5, 4.5, 7.0, 9.3, 11.6, 13.6]
    for xs, xe in zip(arrow_starts, arrow_ends):
        ax.annotate('', xy=(xe, 3.5), xytext=(xs, 3.5),
                    arrowprops=dict(arrowstyle='->', lw=2, color='black'))
    
    tap_layers = [3, 6, 9, 12]
    for i, layer in enumerate(tap_layers):
        x_tap = 4.7 + i * 0.4
        ax.annotate('', xy=(7.9, 2.5), xytext=(x_tap, 2.5),
                    arrowprops=dict(arrowstyle='->', lw=1, color='gray',
                                   connectionstyle='arc3,rad=0.3'))
        ax.text(x_tap, 2.2, f'L{layer}', fontsize=7, ha='center', color='gray')
    
    ax.set_title('DPT Architecture: ViT → Reassemble → Fusion → Depth', fontsize=14)
    plt.tight_layout()
    plt.show()

visualize_dpt_architecture()

---
## 4. Self-Attention Mathematics

### 4.1 Queries, Keys, and Values

Given an input sequence $X \in \mathbb{R}^{N \times D}$ (where $N$ is the number
of tokens and $D$ the embedding dimension), self-attention computes three
projections:

$$
Q = X W_Q, \qquad K = X W_K, \qquad V = X W_V
$$

where $W_Q, W_K \in \mathbb{R}^{D \times d_k}$ and $W_V \in \mathbb{R}^{D \times d_v}$.

**Intuition**:
- **Query** $Q$: "What am I looking for?"
- **Key** $K$: "What do I contain?"
- **Value** $V$: "What information do I carry?"

### 4.2 Scaled Dot-Product Attention

$$
\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^T}{\sqrt{d_k}}\right) V
$$

**Step by step**:

1. **Similarity scores**: $S = Q K^T \in \mathbb{R}^{N \times N}$
   - $S_{ij} = \mathbf{q}_i^T \mathbf{k}_j$ measures how much token $i$ attends to token $j$

2. **Scaling**: $\tilde{S} = S / \sqrt{d_k}$
   - Without scaling, for large $d_k$ the dot products become large, pushing
     softmax into saturation (near-zero gradients)
   - If $q, k$ are i.i.d. with variance 1, then $\text{Var}(q^T k) = d_k$.
     Dividing by $\sqrt{d_k}$ restores unit variance.

3. **Softmax**: $A = \text{softmax}(\tilde{S})$
   - Row-wise softmax: $A_{ij} = \frac{\exp(\tilde{S}_{ij})}{\sum_k \exp(\tilde{S}_{ik})}$
   - Each row sums to 1 → attention **weights**

4. **Weighted aggregation**: $\text{Output} = A \, V \in \mathbb{R}^{N \times d_v}$
   - Each output token is a weighted sum of all value vectors

### 4.3 Multi-Head Attention

Instead of a single attention function, use $h$ parallel **heads** with
different projections:

$$
\text{head}_i = \text{Attention}(X W_Q^{(i)}, X W_K^{(i)}, X W_V^{(i)})
$$

Concatenate and project:

$$
\text{MHSA}(X) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) \, W_O
$$

where $W_O \in \mathbb{R}^{(h \cdot d_v) \times D}$.

Typically $d_k = d_v = D / h$, so the total computation is comparable to
single-head attention with full dimensionality.

### 4.4 Computational Complexity

Self-attention has complexity $O(N^2 \cdot D)$ — quadratic in the sequence
length $N$. For an image of size $224 \times 224$ with patch size 16:

$$
N = \frac{224}{16} \times \frac{224}{16} = 14 \times 14 = 196 \text{ tokens}
$$

The $196 \times 196$ attention matrix is manageable. Larger images or smaller
patches (higher resolution) require efficient attention variants.

In [ ]:
def scaled_dot_product_attention(Q, K, V):
    """
    Compute scaled dot-product attention.
    
    Attention(Q, K, V) = softmax(Q K^T / sqrt(d_k)) V
    """
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    
    # Numerically stable softmax
    scores_shifted = scores - scores.max(axis=-1, keepdims=True)
    exp_scores = np.exp(scores_shifted)
    attention_weights = exp_scores / exp_scores.sum(axis=-1, keepdims=True)
    
    output = attention_weights @ V
    return output, attention_weights


def multi_head_attention(X, W_Q_heads, W_K_heads, W_V_heads, W_O):
    """Multi-head self-attention."""
    h = len(W_Q_heads)
    heads = []
    all_weights = []
    
    for i in range(h):
        Q = X @ W_Q_heads[i]
        K = X @ W_K_heads[i]
        V = X @ W_V_heads[i]
        head_out, attn_w = scaled_dot_product_attention(Q, K, V)
        heads.append(head_out)
        all_weights.append(attn_w)
    
    concat = np.concatenate(heads, axis=-1)
    output = concat @ W_O
    return output, all_weights


N, D = 8, 32  # 8 tokens, 32-dim embeddings
h = 4         # 4 attention heads
d_k = D // h  # 8-dim per head

rng = np.random.RandomState(42)
X = rng.randn(N, D).astype(np.float32)

W_Q_heads = [rng.randn(D, d_k).astype(np.float32) * 0.1 for _ in range(h)]
W_K_heads = [rng.randn(D, d_k).astype(np.float32) * 0.1 for _ in range(h)]
W_V_heads = [rng.randn(D, d_k).astype(np.float32) * 0.1 for _ in range(h)]
W_O = rng.randn(h * d_k, D).astype(np.float32) * 0.1

output, all_weights = multi_head_attention(X, W_Q_heads, W_K_heads, W_V_heads, W_O)

print(f"Input X:          shape {X.shape}")
print(f"Output:           shape {output.shape}")
print(f"Attention weights: {h} heads, each {all_weights[0].shape}")
print(f"\nRow sums of attention weights (should all be 1.0):")
for i, w in enumerate(all_weights):
    print(f"  Head {i}: {w.sum(axis=-1)}")

In [ ]:
fig, axes = plt.subplots(1, h, figsize=(16, 4))
for i in range(h):
    im = axes[i].imshow(all_weights[i], cmap='viridis', vmin=0, vmax=0.5)
    axes[i].set_title(f'Head {i+1}', fontsize=12)
    axes[i].set_xlabel('Key (token j)')
    if i == 0:
        axes[i].set_ylabel('Query (token i)')
    plt.colorbar(im, ax=axes[i], shrink=0.8)
plt.suptitle('Multi-Head Attention Weights', fontsize=14)
plt.tight_layout()
plt.show()

---
## 4b. MiDaS — Mixed Data Sampling (Ranftl et al., 2020)

### The Foundation for Modern Monocular Depth

Before Depth Anything, **MiDaS** established the paradigm of training a single
depth network on a **mixture of diverse depth datasets** — each with different
depth representations, scales, and domains.

### 4b.1 The Core Problem

Different depth datasets use incompatible depth scales and formats:

| Dataset | Domain | Depth type |
|:---|:---|:---|
| MegaDepth | Outdoor landmarks | SfM sparse depth |
| HRWSI | Web stereo images | Disparity |
| ReDWeb | Diverse web images | Relative depth |
| DIML | Indoor/outdoor | Kinect depth |

Naïvely mixing them fails because a loss like MSE would penalise the network
for predicting "wrong" scale — even if the **relative ordering** is perfect.

### 4b.2 Affine-Invariant Loss

MiDaS solves this with an **affine-invariant loss** that aligns each prediction
to its ground truth before computing the error:

$$
L = \frac{1}{N}\sum_{i=1}^{N} \bigl(s \cdot d_i^* + o - d_i\bigr)^2
$$

where $d_i^*$ is the predicted depth, $d_i$ is the GT depth, and the per-image
**scale** $s$ and **offset** $o$ are computed in closed form via median alignment:

$$
s = \frac{\text{median}(|d_i - \text{median}(d_i)|)}{\text{median}(|d_i^* - \text{median}(d_i^*)|)}, \qquad
o = \text{median}(d_i) - s \cdot \text{median}(d_i^*)
$$

This is exactly the scale/shift alignment from Section 2, applied **inside the
training loss**. By making the loss invariant to per-dataset scale and shift,
MiDaS learns **robust relative depth** that generalises across domains.

### 4b.3 Proof of Affine Invariance

**Claim.** The loss $L = \min_{s,o} \frac{1}{N}\sum_i (s \cdot d_i^* + o - d_i)^2$ is
invariant to affine reparametrisations of the prediction $d^*$.

**Proof.** Suppose the network output is transformed: $\tilde{d}_i^* = a \cdot d_i^* + b$
for arbitrary $a \neq 0$, $b \in \mathbb{R}$.  The loss under the transformed prediction is:

$$\tilde{L} = \min_{\tilde{s},\tilde{o}} \frac{1}{N}\sum_i \bigl(\tilde{s}(a d_i^* + b) + \tilde{o} - d_i\bigr)^2
= \min_{\tilde{s},\tilde{o}} \frac{1}{N}\sum_i \bigl((\tilde{s}a) d_i^* + (\tilde{s}b + \tilde{o}) - d_i\bigr)^2$$

Substituting $s' = \tilde{s}a$ and $o' = \tilde{s}b + \tilde{o}$:

$$\tilde{L} = \min_{s', o'} \frac{1}{N}\sum_i (s' d_i^* + o' - d_i)^2 = L$$

The substitution is invertible (given $a \neq 0$: $\tilde{s} = s'/a$,
$\tilde{o} = o' - bs'/a$), so the minimisation over $(\tilde{s}, \tilde{o})$
and $(s', o')$ ranges over the same set of achievable residuals.  Therefore
$\tilde{L} = L$. $\square$

**Consequence.** Since $L$ depends only on the structure of $d^*$ that *cannot* be
explained by any affine alignment, MiDaS never needs to learn absolute scale — it
focuses on geometric structure (relative ordering, surface shape). This is why it
generalises across datasets with incompatible depth scales.

### 4b.4 From MiDaS to Depth Anything

MiDaS proved the approach works. Depth Anything v2 builds directly on this
foundation:

| Aspect | MiDaS | Depth Anything v2 |
|:---|:---|:---|
| Training data | ~2M images, multiple depth datasets | Synthetic pre-train + massive unlabeled real data |
| Backbone | ViT (via DPT) | DINOv2 + DPT |
| Unlabeled data | Not used | Pseudo-labels from teacher on 62M images |
| Scale handling | Affine-invariant loss | Same principle + synthetic data with true scale |

The progression: MiDaS showed that mixing datasets with affine-invariant
training produces robust monocular depth. Depth Anything scaled this further
by adding self-supervised pre-training and massive pseudo-labeled data.

---
## 5. Depth Anything v2 / v3 Overview

### 5.1 Architecture

**Depth Anything** is a family of monocular depth estimators that combine:

1. **DINOv2 backbone** — a self-supervised ViT trained with self-distillation
   on 142M images. Produces powerful, general-purpose visual features.

2. **DPT head** — the reassemble + fusion architecture described above, adapted
   to the DINOv2 token structure.

### 5.2 Training Strategy

**Depth Anything v2** improvements over v1:

- **Synthetic data pre-training**: train on large-scale synthetic datasets
  (with perfect depth GT) to learn precise geometry
- **Real data fine-tuning**: unlabeled real images with pseudo-labels from
  a teacher model
- **Scale-invariant loss** during training (Section 8)

### 5.3 Output Semantics

Depth Anything produces **affine-invariant** inverse depth:

$$
\hat{d}_{\text{rel}} = \frac{1}{\alpha Z + \beta}
$$

To obtain metric depth, you need external calibration or the **Depth Anything
Metric** variant, which is fine-tuned on specific domains (indoor/outdoor) with
absolute depth labels:

$$
\hat{Z}_{\text{metric}} = \frac{d_{\min} \cdot d_{\max}}{\hat{d}_{\text{rel}} \cdot (d_{\max} - d_{\min}) + d_{\min}}
$$

where $d_{\max}$ and $d_{\min}$ are the bounds of the metric depth range for the target domain (e.g., 0.1–10 m for indoor, 0.1–80 m for outdoor).

### 5.4 Model Variants

| Model | Backbone | Params | Speed |
|:---|:---|:---:|:---|
| DA v2 Small | ViT-S/14 | 25M | Real-time (50+ FPS) |
| DA v2 Base | ViT-B/14 | 98M | ~30 FPS |
| DA v2 Large | ViT-L/14 | 335M | ~15 FPS |
| DA v2 Giant | ViT-G/14 | 1.3B | ~5 FPS |

In [ ]:
def simulate_depth_anything_pipeline(depth_gt, H=240, W=320):
    """Simulate the Depth Anything pipeline on synthetic data."""
    patch_size = 16
    n_h = H // patch_size
    n_w = W // patch_size
    n_tokens = n_h * n_w
    D_embed = 384  # ViT-S dimension
    
    rng = np.random.RandomState(42)
    tokens = rng.randn(n_tokens, D_embed).astype(np.float32)
    
    d_pred_coarse = np.mean(tokens[:, :3], axis=1)
    d_pred_coarse = d_pred_coarse.reshape(n_h, n_w)
    
    d_pred = cv2.resize(d_pred_coarse, (W, H), interpolation=cv2.INTER_LINEAR)
    
    d_pred_normed = (d_pred - d_pred.min()) / (d_pred.max() - d_pred.min() + 1e-8)
    
    alpha_sim = depth_gt.max() - depth_gt.min()
    beta_sim = depth_gt.min()
    d_pred_metric = d_pred_normed * alpha_sim + beta_sim
    
    noise = rng.randn(H, W).astype(np.float32) * 0.5
    d_pred_metric = gaussian_filter(d_pred_metric + noise, sigma=3)
    
    print(f"Image size: {H}×{W}")
    print(f"Patch size: {patch_size}")
    print(f"Token grid: {n_h}×{n_w} = {n_tokens} tokens")
    print(f"Embedding dim: {D_embed}")
    
    return d_pred_metric

d_pred_sim = simulate_depth_anything_pipeline(depth_gt)

### 5.5 Apple Depth Pro & UniDepthV2

Two alternative foundation models that push the boundaries in different
directions:

**Apple Depth Pro (2024)**

- Produces sharp **2.25-megapixel metric depth** in 0.3 seconds
- Multi-scale ViT architecture with boundary-aware training
- State-of-the-art **boundary sharpness** (dedicated F1 boundary metric)
- Also estimates **focal length** from the image — no camera intrinsics needed
- Particularly strong on close-range, high-resolution applications

**UniDepthV2 (2024)**

- Predicts a dense 3D point in metric space per pixel from RGB alone
- Self-promptable camera module learns a dense camera representation
  without requiring intrinsics as input
- Pseudo-spherical output space **disentangles camera and depth**:

$$
\hat{\mathbf{X}} = \hat{Z} \cdot \begin{pmatrix} \sin\hat{\theta}\cos\hat{\phi} \\ \sin\hat{\theta}\sin\hat{\phi} \\ \cos\hat{\theta} \end{pmatrix}
$$

- **#1 on KITTI depth benchmark** (at time of publication)
- Produces **per-pixel uncertainty** estimates useful for downstream TSDF
  fusion (NB 12) — uncertain pixels get lower weight

### Practical Comparison

| Model | Metric | Boundary | Speed | Intrinsics? |
|-------|:------:|:--------:|:-----:|:-----------:|
| DA v2 Large | Relative | Good | 15 FPS | Required |
| DA v2 Metric | Absolute | Good | 15 FPS | Required |
| Depth Pro | Absolute | **Best** | 3 FPS | Not needed |
| UniDepthV2 | Absolute | Good | 8 FPS | Not needed |

**Takeaway**: For drones, DA v2 Small/Base (metric variant) gives the best
speed-accuracy trade-off. Depth Pro wins when boundary precision matters
(e.g., grasping, augmented reality). UniDepthV2's uncertainty output makes
it ideal for TSDF fusion pipelines.

### 5.6 Depth Anything v3 (Lin et al., ICLR 2026)

While DA v2 excels at **monocular** relative depth, it cannot reason about
multi-view geometry — it processes each image independently. **Depth Anything
v3 (DA3)** unifies mono and multi-view depth estimation in a single model.

#### The Depth-Ray Representation

DA3 predicts not just depth but also **ray direction** for each pixel:

$$
(\hat{d}_i, \hat{\mathbf{r}}_i) = f_\theta(I, i) \qquad \text{where } \hat{\mathbf{r}}_i \in \mathbb{S}^2
$$

Given an image pixel $i$, the 3D point is recovered as:

$$
\hat{\mathbf{X}}_i = \hat{d}_i \cdot \hat{\mathbf{r}}_i
$$

This **disentangles depth from camera intrinsics** — the ray encodes the
camera model implicitly, so no calibration parameters are needed at inference.

#### Architecture

- **Single plain ViT encoder** — no separate pose estimation network
- Multi-view images are tokenised and processed jointly via self-attention
- For mono input: degrades gracefully to standard monocular depth
- For multi-view input: cross-image attention captures epipolar relationships

#### Key Results

| Metric | DA3 vs VGGT | Improvement |
|:---|:---|:---:|
| Pose accuracy (rotation + translation) | DA3 >> VGGT | **35.7%** |
| Geometric accuracy (depth + reconstruction) | DA3 >> VGGT | **23.6%** |

#### 3D Gaussian Prediction

DA3 can also predict **per-pixel 3D Gaussians** (position, covariance, colour,
opacity) for novel view synthesis — bridging depth estimation and 3D Gaussian
Splatting in a single forward pass.

#### Significance

DA3 represents the **convergence of depth estimation and multi-view geometry**.
Rather than treating monocular depth and SfM/MVS as separate problems (NB 10),
DA3 handles both in a unified framework: one image gives monocular depth,
multiple images give calibrated metric depth + camera poses.

---
## 6. DUSt3R / MASt3R: Predicting 3D Pointmaps Directly

### 6.1 A Paradigm Shift

Traditional 3D reconstruction requires:
1. Camera intrinsics calibration
2. Feature matching
3. Pose estimation
4. Triangulation

**DUSt3R** (Dense and Unconstrained Stereo 3D Reconstruction) **bypasses all
of this** by directly predicting 3D pointmaps from image pairs.

### 6.2 DUSt3R Architecture

Given two images $I_1, I_2$:

$$
\text{DUSt3R}(I_1, I_2) \rightarrow (\mathbf{X}^{(1)}, \mathbf{X}^{(2)}, C^{(1)}, C^{(2)})
$$

where:
- $\mathbf{X}^{(v)} \in \mathbb{R}^{H \times W \times 3}$: predicted 3D pointmap
  for view $v$ **in the coordinate frame of view 1**
- $C^{(v)} \in \mathbb{R}^{H \times W}$: per-pixel confidence

The model uses:
- A shared ViT encoder for both images
- Cross-attention decoders that exchange information between views
- Regression heads that output 3D coordinates directly

### 6.3 Training Loss

Confidence-weighted 3D regression loss:

$$
\mathcal{L} = \sum_{v \in \{1,2\}} \sum_{i} C_i^{(v)} \cdot \|\hat{\mathbf{X}}_i^{(v)} - \mathbf{X}_i^{(v)}\|_1 - \alpha \log C_i^{(v)}
$$

### 6.4 MASt3R Extension

**MASt3R** (Matching And Stereo 3D Reconstruction) extends DUSt3R with:
- Dense local feature prediction for matching
- Improved multi-view global alignment
- Better handling of wide-baseline pairs

In [ ]:
def compare_pipelines():
    """Compare traditional vs DUSt3R pipeline."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
    
    trad_steps = [
        (0.5, 5, 'Calibrate\nCameras'),
        (0.5, 4, 'Detect\nFeatures'),
        (0.5, 3, 'Match\nFeatures'),
        (0.5, 2, 'Estimate\nPoses'),
        (0.5, 1, 'Triangulate\n3D Points'),
    ]
    for x, y, text in trad_steps:
        rect = plt.Rectangle((x-0.4, y-0.4), 0.8, 0.8, linewidth=2,
                             edgecolor='black', facecolor='lightblue', zorder=2)
        ax1.add_patch(rect)
        ax1.text(x, y, text, ha='center', va='center', fontsize=9, fontweight='bold')
    for i in range(len(trad_steps) - 1):
        ax1.annotate('', xy=(0.5, trad_steps[i+1][1] + 0.4),
                     xytext=(0.5, trad_steps[i][1] - 0.4),
                     arrowprops=dict(arrowstyle='->', lw=2))
    ax1.set_xlim(-0.5, 1.5)
    ax1.set_ylim(0, 6)
    ax1.set_title('Traditional Pipeline', fontsize=13)
    ax1.axis('off')
    
    dust3r_steps = [
        (0.5, 4, 'Image Pair\nInput'),
        (0.5, 2.5, 'DUSt3R\nNeural Network'),
        (0.5, 1, '3D Pointmaps\n+ Confidence'),
    ]
    for x, y, text in dust3r_steps:
        rect = plt.Rectangle((x-0.4, y-0.4), 0.8, 0.8, linewidth=2,
                             edgecolor='black', facecolor='lightgreen', zorder=2)
        ax2.add_patch(rect)
        ax2.text(x, y, text, ha='center', va='center', fontsize=9, fontweight='bold')
    for i in range(len(dust3r_steps) - 1):
        ax2.annotate('', xy=(0.5, dust3r_steps[i+1][1] + 0.4),
                     xytext=(0.5, dust3r_steps[i][1] - 0.4),
                     arrowprops=dict(arrowstyle='->', lw=2))
    ax2.set_xlim(-0.5, 1.5)
    ax2.set_ylim(0, 6)
    ax2.set_title('DUSt3R Pipeline', fontsize=13)
    ax2.axis('off')
    
    plt.suptitle('Traditional vs Neural 3D Reconstruction', fontsize=14)
    plt.tight_layout()
    plt.show()

compare_pipelines()

---
## 7. Depth Evaluation Metrics

### 7.1 Standard Metrics

Given predicted depth $\hat{d}_i$ and ground truth $d_i^*$ at $N$ valid pixels:

**Error metrics** (lower is better):

$$
\text{abs\_rel} = \frac{1}{N} \sum_{i=1}^{N} \frac{|\hat{d}_i - d_i^*|}{d_i^*}
$$

$$
\text{sq\_rel} = \frac{1}{N} \sum_{i=1}^{N} \frac{(\hat{d}_i - d_i^*)^2}{d_i^*}
$$

$$
\text{RMSE} = \sqrt{\frac{1}{N} \sum_{i=1}^{N} (\hat{d}_i - d_i^*)^2}
$$

$$
\text{RMSE}_{\log} = \sqrt{\frac{1}{N} \sum_{i=1}^{N} (\log \hat{d}_i - \log d_i^*)^2}
$$

**Accuracy metrics** (higher is better) — **threshold accuracy**:

$$
\delta_k = \frac{1}{N} \left| \left\{ i : \max\!\left(\frac{\hat{d}_i}{d_i^*}, \frac{d_i^*}{\hat{d}_i}\right) < 1.25^k \right\} \right|
$$

for $k \in \{1, 2, 3\}$.

**Interpretation of $\delta_1$**: the percentage of pixels where the predicted
depth is within 25% of the ground truth (ratio between 0.8 and 1.25).

### 7.2 Why These Specific Thresholds?

The $\delta$ thresholds form a **geometric progression**:

$$
\delta_1: \quad \frac{\hat{d}}{d^*} \in [0.800, 1.250] \quad (\pm 25\%)
$$
$$
\delta_2: \quad \frac{\hat{d}}{d^*} \in [0.640, 1.5625] \quad (\pm 56\%)
$$
$$
\delta_3: \quad \frac{\hat{d}}{d^*} \in [0.512, 1.9531] \quad (\pm 95\%)
$$

Good models achieve $\delta_1 > 0.95$, meaning 95%+ of pixels are within
25% of the true depth.

In [ ]:
def compute_depth_metrics(predicted, ground_truth, mask=None):
    """Compute all standard depth evaluation metrics."""
    if mask is None:
        mask = ground_truth > 0
    
    pred = np.clip(predicted[mask].astype(np.float64), 1e-6, None)
    gt = ground_truth[mask].astype(np.float64)
    
    N = len(pred)
    if N == 0:
        return {}
    
    diff = np.abs(pred - gt)
    
    abs_rel = float(np.mean(diff / gt))
    sq_rel = float(np.mean(diff**2 / gt))
    rmse = float(np.sqrt(np.mean(diff**2)))
    rmse_log = float(np.sqrt(np.mean((np.log(pred) - np.log(gt))**2)))
    
    ratio = np.maximum(pred / gt, gt / pred)
    delta_1 = float(np.mean(ratio < 1.25))
    delta_2 = float(np.mean(ratio < 1.25**2))
    delta_3 = float(np.mean(ratio < 1.25**3))
    
    return {
        'abs_rel': abs_rel,
        'sq_rel': sq_rel,
        'rmse': rmse,
        'rmse_log': rmse_log,
        'delta_1': delta_1,
        'delta_2': delta_2,
        'delta_3': delta_3,
    }


def generate_noisy_prediction(depth_gt, noise_level=0.15, seed=42):
    """Add multiplicative noise to simulate a network prediction."""
    rng = np.random.RandomState(seed)
    noise = 1.0 + rng.randn(*depth_gt.shape) * noise_level
    return depth_gt * noise


print("=== Depth Metrics on Synthetic Data ===")
print()

for noise_level, label in [(0.05, 'Low noise'), (0.15, 'Medium noise'), (0.30, 'High noise')]:
    d_pred = generate_noisy_prediction(depth_gt, noise_level=noise_level)
    metrics = compute_depth_metrics(d_pred, depth_gt)
    
    print(f"--- {label} (σ = {noise_level}) ---")
    print(f"  abs_rel:  {metrics['abs_rel']:.4f}")
    print(f"  sq_rel:   {metrics['sq_rel']:.4f}")
    print(f"  RMSE:     {metrics['rmse']:.4f} m")
    print(f"  RMSE_log: {metrics['rmse_log']:.4f}")
    print(f"  δ₁:       {metrics['delta_1']:.4f}  ({metrics['delta_1']*100:.1f}%)")
    print(f"  δ₂:       {metrics['delta_2']:.4f}  ({metrics['delta_2']*100:.1f}%)")
    print(f"  δ₃:       {metrics['delta_3']:.4f}  ({metrics['delta_3']*100:.1f}%)")
    print()

In [ ]:
noise_levels = np.linspace(0.01, 0.5, 50)
metric_names = ['abs_rel', 'rmse', 'delta_1', 'delta_2', 'delta_3']
results = {name: [] for name in metric_names}

for nl in noise_levels:
    d_pred = generate_noisy_prediction(depth_gt, noise_level=nl)
    m = compute_depth_metrics(d_pred, depth_gt)
    for name in metric_names:
        results[name].append(m[name])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(noise_levels, results['abs_rel'], 'b-', label='abs_rel', linewidth=2)
ax1.plot(noise_levels, np.array(results['rmse']) / depth_gt.max(), 'r-',
         label='RMSE / max_depth', linewidth=2)
ax1.set_xlabel('Noise Level σ')
ax1.set_ylabel('Error')
ax1.set_title('Error Metrics vs Noise Level')
ax1.legend()
ax1.grid(True, alpha=0.3)

for name, style in [('delta_1', '-'), ('delta_2', '--'), ('delta_3', ':')]:
    ax2.plot(noise_levels, results[name], style, label=name, linewidth=2)
ax2.set_xlabel('Noise Level σ')
ax2.set_ylabel('Accuracy')
ax2.set_title('Threshold Accuracy vs Noise Level')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

---
## 8. Loss Functions for Depth Training

### 8.1 Scale-Invariant Loss (Eigen et al., 2014)

The scale-invariant loss operates in log-depth space and is invariant to global
scale changes:

$$
\mathcal{L}_{\text{SI}} = \frac{1}{N} \sum_i e_i^2 - \frac{\lambda}{N^2} \left(\sum_i e_i\right)^2
$$

where $e_i = \log \hat{d}_i - \log d_i^*$ and $\lambda \in [0, 1]$.

**Breakdown of the two terms**:

1. $\frac{1}{N} \sum_i e_i^2$: standard MSE in log-depth space
2. $\frac{\lambda}{N^2} (\sum_i e_i)^2 = \lambda \bar{e}^2$: subtracts the **squared mean error**,
   making the loss invariant to a global additive offset in log-depth
   (= multiplicative scale in linear depth). Derivation: $(\sum e_i)^2 = N^2 \bar{e}^2$,
   so $\lambda / N^2 \cdot N^2 \bar{e}^2 = \lambda \bar{e}^2$.
   When subtracted from the MSE, the result is the variance of $\mathbf{e}$.

**Special cases of $\lambda$:**
- $\lambda = 0$: reduces to **log-MSE** $= \frac{1}{N}\sum_i e_i^2$ — penalises absolute log-depth error with no scale-invariance.
- $\lambda = 1$: the loss is fully **scale-invariant** (Eigen et al., 2014):

$$
\mathcal{L}_{\text{SI}}^{\lambda=1} = \text{Var}(\mathbf{e}) = \frac{1}{N}\sum_i (e_i - \bar{e})^2
$$

because $\frac{1}{N}\sum e_i^2 - \frac{1}{N^2}(\sum e_i)^2 = \frac{1}{N}\sum e_i^2 - \bar{e}^2 = \text{Var}(e)$.

A global scale change $\hat{d} \to s \cdot \hat{d}$ adds a constant $\log s$ to
every $e_i$, shifting $\bar{e}$ by the same amount but leaving $\text{Var}(e)$
unchanged — hence scale-invariance.

### Gradient of $\mathcal{L}_{\text{SI}}$ w.r.t. $\hat{d}_i$

$$
\frac{\partial \mathcal{L}_{\text{SI}}}{\partial \hat{d}_i}
= \frac{\partial \mathcal{L}}{\partial e_i} \cdot \frac{\partial e_i}{\partial \hat{d}_i}
$$

Since $e_i = \log \hat{d}_i - \log d_i^*$, we have $\frac{\partial e_i}{\partial \hat{d}_i} = \frac{1}{\hat{d}_i}$. For the loss terms:

$$
\frac{\partial}{\partial e_i}\left[\frac{1}{N}\sum_k e_k^2\right] = \frac{2 e_i}{N}, \qquad
\frac{\partial}{\partial e_i}\left[\frac{\lambda}{N^2}\left(\sum_k e_k\right)^2\right] = \frac{2\lambda}{N^2}\sum_k e_k
$$

Combining:

$$
\boxed{\frac{\partial \mathcal{L}_{\text{SI}}}{\partial \hat{d}_i}
= \frac{1}{\hat{d}_i}\left(\frac{2 e_i}{N} - \frac{2\lambda}{N^2}\sum_j e_j\right)}
$$

**Interpretation**: the gradient has two components — a per-pixel term proportional
to $e_i$ that pulls each prediction toward its target, minus a global mean-correction
term (scaled by $\lambda$) that removes the shared scale error from all gradients.
When $\lambda = 1$ the gradient becomes $\frac{2}{\hat{d}_i N}(e_i - \bar{e})$,
so the network only receives gradients from the *relative* structure of the errors.

**Proof of scale invariance.** Under a global scale change $\hat{d}_i \to \alpha\,\hat{d}_i$, each $e_i \to e_i + \log\alpha$, so $\bar{e} \to \bar{e} + \log\alpha$, and $(e_i - \bar{e})$ is unchanged. Hence $\text{Var}(\mathbf{e})$ is invariant. $\square$

### 8.2 Gradient Matching Loss

Encourages correct depth **edges** and **relative ordering**:

$$
\mathcal{L}_{\text{grad}} = \frac{1}{N} \sum_i \left[
\left|\frac{\partial e_i}{\partial x}\right| + \left|\frac{\partial e_i}{\partial y}\right|
\right]
$$

where $\frac{\partial e}{\partial x}$ and $\frac{\partial e}{\partial y}$ are
the horizontal and vertical gradients of the log-depth error.

Equivalently, in multi-scale form:

$$
\mathcal{L}_{\text{grad}}^{\text{multi}} = \sum_{s=1}^{S} \frac{1}{N_s} \sum_i
\left[
  \left|\nabla_x^{(s)} \hat{d}_i - \nabla_x^{(s)} d_i^*\right|
  + \left|\nabla_y^{(s)} \hat{d}_i - \nabla_y^{(s)} d_i^*\right|
\right]
$$

### 8.3 Combined Loss

Modern depth networks typically combine multiple losses:

$$
\mathcal{L}_{\text{total}} = w_1 \mathcal{L}_{\text{SI}} + w_2 \mathcal{L}_{\text{grad}} + w_3 \mathcal{L}_{\text{SSIM}}
$$

In [ ]:
def scale_invariant_loss(pred, gt, lam=0.5, mask=None):
    """
    Scale-invariant loss (Eigen et al., 2014).
    
    L_SI = (1/N) Σ eᵢ² - (λ/N²)(Σ eᵢ)²
    where eᵢ = log(d_pred) - log(d_gt)
    """
    if mask is None:
        mask = (gt > 0) & (pred > 0)
    
    log_pred = np.log(np.clip(pred[mask], 1e-6, None))
    log_gt = np.log(np.clip(gt[mask], 1e-6, None))
    e = log_pred - log_gt
    N = len(e)
    
    loss = np.mean(e**2) - lam * (np.sum(e)**2) / (N**2)
    return float(loss)


def gradient_matching_loss(pred, gt, mask=None):
    """
    Gradient matching loss.
    
    L_grad = (1/N) Σ (|∂e/∂x| + |∂e/∂y|)
    """
    if mask is None:
        mask = (gt > 0) & (pred > 0)
    
    log_pred = np.log(np.clip(pred, 1e-6, None))
    log_gt = np.log(np.clip(gt, 1e-6, None))
    e = log_pred - log_gt
    
    grad_x = sobel(e, axis=1)
    grad_y = sobel(e, axis=0)
    
    loss = np.mean(np.abs(grad_x[mask]) + np.abs(grad_y[mask]))
    return float(loss)


print("=== Loss Functions ===")
print()

for noise_level, label in [(0.05, 'Low'), (0.15, 'Medium'), (0.30, 'High')]:
    d_pred = generate_noisy_prediction(depth_gt, noise_level=noise_level)
    si_loss = scale_invariant_loss(d_pred, depth_gt, lam=0.5)
    grad_loss = gradient_matching_loss(d_pred, depth_gt)
    print(f"{label} noise (σ={noise_level}):  SI loss = {si_loss:.6f},  Grad loss = {grad_loss:.6f}")

print()
print("=== Scale Invariance Demonstration ===")
d_scaled = depth_gt * 2.0  # 2× scale
print(f"SI loss (λ=1.0) with 2× scaled GT: {scale_invariant_loss(d_scaled, depth_gt, lam=1.0):.8f}")
print("(Should be ~0 because SI loss with λ=1 is invariant to global scale)")

In [ ]:
lambdas = np.linspace(0, 1, 50)
noise_configs = [(0.1, 'blue'), (0.2, 'red'), (0.3, 'green')]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for nl, color in noise_configs:
    d_pred = generate_noisy_prediction(depth_gt, noise_level=nl)
    losses = [scale_invariant_loss(d_pred, depth_gt, lam=l) for l in lambdas]
    ax1.plot(lambdas, losses, color=color, linewidth=2, label=f'σ = {nl}')

ax1.set_xlabel('λ')
ax1.set_ylabel('SI Loss')
ax1.set_title('Scale-Invariant Loss vs λ')
ax1.legend()
ax1.grid(True, alpha=0.3)

d_pred_med = generate_noisy_prediction(depth_gt, noise_level=0.15)
log_error = np.log(np.clip(d_pred_med, 1e-6, None)) - np.log(np.clip(depth_gt, 1e-6, None))
grad_x = sobel(log_error, axis=1)
grad_y = sobel(log_error, axis=0)
grad_mag = np.sqrt(grad_x**2 + grad_y**2)

im = ax2.imshow(grad_mag, cmap='hot', vmin=0, vmax=1)
ax2.set_title('Log-Depth Error Gradient Magnitude')
ax2.axis('off')
plt.colorbar(im, ax=ax2, shrink=0.8)

plt.tight_layout()
plt.show()

## 9. Self-Supervised Depth Training (Monodepth2)

### The Key Insight
You don't need ground-truth depth to train a depth network. Given three
consecutive frames $(I_{t-1}, I_t, I_{t+1})$, simultaneously train:
- **DepthNet**: predicts depth $D_t$ for $I_t$
- **PoseNet**: predicts relative pose $T_{t \to t'}$

### Differentiable Warping

For each pixel $\mathbf{p} = (u, v, 1)^T$ in the target frame $I_t$ with predicted
depth $D_t(\mathbf{p})$, the corresponding pixel $\mathbf{p}'$ in the source frame
$I_{t'}$ is computed by:

$$
\boxed{\mathbf{p}' \sim K \, T_{t \to t'} \, D_t(\mathbf{p}) \, K^{-1} \mathbf{p}}
$$

where $K$ is the camera intrinsic matrix, $T_{t \to t'} \in SE(3)$ is the predicted
relative pose, and $\sim$ denotes equality up to projective normalisation
(divide by the third coordinate). This is the standard unproject → transform → reproject
chain: $K^{-1}\mathbf{p}$ lifts the pixel to a unit ray, scaling by $D_t(\mathbf{p})$
gives the 3D point, $T_{t \to t'}$ moves it to the source frame, and $K$ reprojects.

The warped image is then formed by bilinear sampling (Spatial Transformer Network):

$$
I_{t' \to t}(\mathbf{p}) = I_{t'}(\mathbf{p}')
$$

This entire pipeline is **differentiable**: bilinear interpolation provides
sub-pixel gradients, and the chain $D_t \to \mathbf{p}' \to I_{t' \to t}$
lets gradients flow back to both DepthNet and PoseNet.

### Photometric Loss

$$
\mathcal{L}_{\text{pe}}(I_a, I_b) = \alpha \cdot \frac{1 - \text{SSIM}(I_a, I_b)}{2}
+ (1 - \alpha) \cdot |I_a - I_b|
$$

with $\alpha = 0.85$ (Monodepth2 default). The SSIM term captures structural
similarity (patches), the L1 term captures per-pixel intensity differences.

### Per-Pixel Minimum Reprojection

With multiple source frames $t' \in \{t-1, t+1\}$, pixels may be visible in one
source but occluded in the other. Instead of averaging (which blurs the loss
signal), Monodepth2 takes the **per-pixel minimum**:

$$
\mathcal{L}_{\text{reproj}}(\mathbf{p}) = \min_{t' \in \{t-1, t+1\}} \mathcal{L}_{\text{pe}}\bigl(I_t(\mathbf{p}),\; I_{t' \to t}(\mathbf{p})\bigr)
$$

This ensures that each pixel's loss is computed against whichever source frame
gives the best reconstruction — effectively handling **occlusion boundaries**
where one source frame lacks the correct correspondence.

### Auto-Masking

In static scenes or when the camera is stationary, warped images can match
the target trivially (the scene hasn't moved). This creates a degenerate
training signal. The **auto-mask** $\mu$ filters out such pixels:

$$
\mu(\mathbf{p}) = \Bigl[\min_{t'} \mathcal{L}_{\text{pe}}(I_t, I_{t' \to t})
\;<\; \min_{t'} \mathcal{L}_{\text{pe}}(I_t, I_{t'})\Bigr]
$$

where $[\cdot]$ is the Iverson bracket. The right-hand side is the photometric
error of the **unwarped** source frame — if warping doesn't reduce the error
compared to the raw source image, the pixel is masked out ($\mu = 0$).

**Why this works**: for a static pixel (e.g., a parked car or the camera not moving),
$I_{t' \to t} \approx I_{t'}$, so the warped and unwarped errors are similar.
The mask suppresses these uninformative pixels, preventing the network from
learning a trivial infinite-depth solution.

### Final Loss

$$
\mathcal{L} = \frac{1}{|\mu|}\sum_{\mathbf{p}} \mu(\mathbf{p}) \cdot \mathcal{L}_{\text{reproj}}(\mathbf{p})
+ \lambda_s \cdot \mathcal{L}_{\text{smooth}}
$$

where $\mathcal{L}_{\text{smooth}}$ is edge-aware depth smoothness:

$$
\mathcal{L}_{\text{smooth}} = |\partial_x d^*| \, e^{-|\partial_x I|} + |\partial_y d^*| \, e^{-|\partial_y I|}
$$

with $d^* = \bar{d} / d$ (mean-normalised inverse depth). Using mean-normalised inverse depth $d^* = \bar{d}/d$ (where $\bar{d}$ is the per-image mean depth) ensures the smoothness loss is scale-invariant — matching the scale-free nature of self-supervised training — and prevents the loss from being dominated by large-depth regions where the absolute gradient would otherwise be small.

### Limitation
Scale ambiguity — self-supervised depth only learns *relative* depth. Metric
depth requires stereo baseline or metric fine-tuning.

### Why This Matters
Understanding self-supervised training explains how **Depth Anything v2** can
train on millions of unlabeled images (pseudo-labels from a strong teacher).

---
## 10. Uncertainty in Depth Estimation

### Why Uncertainty Matters
For TSDF fusion (NB 12), knowing which pixels are **confident** and which
are **unreliable** lets you weight measurements appropriately.

### Methods

| Method | Cost | Quality | Notes |
|--------|------|---------|-------|
| **GNLL** (learned variance) | 1× | Good | Add log-variance output head |
| **MC Dropout** | N× | Good | Keep dropout at test time, N forward passes |
| **Test-time augmentation** | K× | Moderate | Flip/crop inputs, compute variance |
| **Ensembles** | K× | Best | Train K independent models |

### GNLL (Gaussian Negative Log-Likelihood)

Add a second output head predicting log-variance $\log\sigma^2$:

$$
\mathcal{L}_{\text{GNLL}} = \frac{1}{2}\left(\log\sigma^2 + \frac{(d - d^*)^2}{\sigma^2}\right)
$$

The network learns to predict **high variance** for ambiguous regions
(reflections, textureless walls).

---
## 11. Video Depth Estimation

### The Problem
Applying Depth Anything per-frame produces depth that **flickers** between
frames — spatially accurate but temporally inconsistent.

### DepthCrafter (CVPR 2025)
Video-to-depth **diffusion model** built from Stable Video Diffusion:
- Processes up to 110 frames at once
- Temporal consistency through video diffusion's learned motion priors
- SOTA zero-shot on Sintel, ScanNet, KITTI, Bonn

### ChronoDepth (CVPR 2025)
Streaming-friendly alternative:
- Sliding window: initialise overlapping frames with previous predictions
- **98% temporal consistency improvement** on KITTI-360
- Better for real-time/online SLAM scenarios

### Why This Matters for Mapping
Temporally consistent depth → smoother TSDF integration → better 3D
reconstructions. A depth model that "agrees with itself" across frames is
practically equivalent to short-baseline stereo.

---
## 12. Surface Normal Estimation

### Why Normals Matter
- Weight TSDF fusion by angle (grazing = unreliable)
- Improve Poisson mesh reconstruction
- Understand scene structure (floor vs wall vs ceiling)

### DSINE (CVPR 2024 Oral)
Camera-intrinsics-aware regression: per-pixel ray direction as input,
piecewise-smooth normals. Fast deterministic inference.

### StableNormal (2024)
Diffusion-based: coarse-to-fine with semantic guidance.
Handles reflective/transparent surfaces. SOTA on DIODE-indoor.

### Connection to Depth
Normals = gradient of depth surface. PatchMatch MVS (NB 10) uses joint
depth+normal hypotheses.

---
## 13. Depth Completion (Discussion)

### The Problem
Sparse depth (from LiDAR, stereo matching, or tracked features) +
dense RGB → dense, accurate depth.

### Why It Matters for Drones
Sparse LiDAR (single-beam) is cheap and light. Combined with a camera,
depth completion gives dense metric depth without heavy sensors.

### Methods
- **SigNet** (CVPR 2025): completion as enhancement — bilateral
  densification + image-guided Mamba network
- **SelfDC** (2025): self-supervised, no dense labels needed
- **StarryGazer** (2025): pseudo-GT from monocular depth models

### Connection to Our Pipeline
In NB 08 (stereo) we get semi-dense disparity; in this notebook, monocular
depth. Depth completion combines **sparse-but-accurate** stereo/LiDAR with
**dense-but-ambiguous** monocular for the best of both worlds.

---
## 14. Exercises

### Exercise 9.1: Implement All Depth Metrics

Complete the function below to compute all seven standard depth metrics.

In [ ]:
def exercise_depth_metrics(predicted, ground_truth, mask=None):
    """
    Exercise 9.1: Implement all standard depth metrics.
    
    Given predicted depth d_hat and ground truth d_gt:
    
    abs_rel  = (1/N) Σ |d_hat - d_gt| / d_gt
    sq_rel   = (1/N) Σ (d_hat - d_gt)² / d_gt
    rmse     = sqrt( (1/N) Σ (d_hat - d_gt)² )
    rmse_log = sqrt( (1/N) Σ (log(d_hat) - log(d_gt))² )
    δ_k      = fraction where max(d_hat/d_gt, d_gt/d_hat) < 1.25^k
    
    Returns: dict with keys abs_rel, sq_rel, rmse, rmse_log, delta_1, delta_2, delta_3
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement depth metrics")


# --- Tests ---
# d_test_gt = np.array([[4.0, 8.0], [2.0, 6.0]])
# d_test_pred = d_test_gt * 1.1  # 10% overestimate
# m = exercise_depth_metrics(d_test_pred, d_test_gt)
# assert abs(m['abs_rel'] - 0.1) < 0.001
# assert m['delta_1'] == 1.0  # all within 25%
# print("✓ exercise_depth_metrics passed")

### Exercise 9.2: Align Predicted Depth to Ground Truth

Implement least-squares alignment to recover optimal $\alpha$ and $\beta$.

In [ ]:
def exercise_align_depth(d_pred, d_gt, mask=None):
    """
    Exercise 9.2: Align predicted depth via least-squares.
    
    Solve: min_{α,β} Σ (α·d_pred + β - d_gt)²
    
    Using normal equations:
    [Σd²   Σd ] [α]   [Σd·g]
    [Σd    N  ] [β] = [Σg   ]
    
    Returns: (alpha, beta, aligned_depth)
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement alignment")


# --- Tests ---
# rng = np.random.RandomState(123)
# d_gt_test = 2.0 + 5.0 * rng.rand(100, 100)
# d_pred_test = (d_gt_test - 1.0) / 3.0 + rng.randn(100, 100) * 0.01
# alpha, beta, aligned = exercise_align_depth(d_pred_test, d_gt_test)
# assert abs(alpha - 3.0) < 0.1, f"Expected α ≈ 3.0, got {alpha:.3f}"
# assert abs(beta - 1.0) < 0.1, f"Expected β ≈ 1.0, got {beta:.3f}"
# print(f"✓ exercise_align_depth passed (α={alpha:.3f}, β={beta:.3f})")

### Exercise 9.3: Scale-Invariant Loss

Implement the scale-invariant loss and verify its invariance property.

In [ ]:
def exercise_si_loss(pred, gt, lam=1.0):
    """
    Exercise 9.3: Scale-invariant loss.
    
    L_SI = (1/N) Σ eᵢ² - (λ/N²)(Σ eᵢ)²
    where eᵢ = log(d_pred) - log(d_gt)
    
    When λ=1, this equals Var(e), which is invariant to global scale.
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement SI loss")


# --- Tests ---
# d_gt_test = np.random.rand(50, 50) * 10 + 1
# d_pred_test = d_gt_test * 3.7  # global scale
# loss_scaled = exercise_si_loss(d_pred_test, d_gt_test, lam=1.0)
# assert loss_scaled < 1e-10, f"SI loss with pure scale should be ~0, got {loss_scaled}"
# print("✓ exercise_si_loss passed: invariant to global scale")

### Exercise 9.4: Full Depth Evaluation Pipeline

Given a synthetic "network output":
1. Align it to the ground truth
2. Compute all metrics before and after alignment
3. Show the improvement

In [ ]:
# Exercise 9.4: Full evaluation pipeline

# Given:
depth_gt_ex = generate_synthetic_depth_gt()
d_network = simulate_network_prediction(depth_gt_ex, scale=0.3, shift=2.0, noise_std=0.2)

# YOUR CODE HERE:
# 1. Compute metrics BEFORE alignment
# 2. Align d_network to depth_gt_ex using least-squares
# 3. Compute metrics AFTER alignment
# 4. Print both sets of metrics
# 5. Visualize: GT, unaligned prediction, aligned prediction, error maps

---

## Limitations & Failure Cases

- **Scale ambiguity:** Monocular depth is inherently relative — the network cannot recover absolute metric scale without calibration data or metric fine-tuning (e.g. ZoeDepth). Two scenes that differ only by a global scale factor produce identical images.
- **Reflective & transparent surfaces:** Mirrors, glass, and water violate the photometric consistency assumptions that depth networks implicitly learn. Reflections create virtual copies of the scene at incorrect depths, and transparency blends foreground and background.
- **Out-of-distribution scenes:** Models trained on indoor/outdoor datasets (NYUv2, KITTI) may fail on unusual domains such as underwater, aerial, medical, or industrial imagery where geometric priors differ substantially.
- **Fine structures:** Thin objects like wires, fences, and railings are often smoothed away or merged with the background because they occupy only a few pixels and are suppressed by the network's receptive field.
- **Moving objects:** Depth estimation assumes a static scene. Moving objects (vehicles, pedestrians) may receive incorrect depth predictions, especially when their motion creates ambiguous scale cues.
- **Temporal flickering:** Per-frame monocular models produce temporally inconsistent depth — the same surface can get different depth values in consecutive frames. This motivates video-consistent methods like DepthCrafter and ChronoDepth that enforce temporal smoothness.

---
## 16. Beyond DPT: Emerging Decoder Architectures (2026)

### The DPT Decoder Bottleneck

DPT's multi-scale **reassemble-then-fuse** strategy (per-layer readout → spatial reassembly
→ cross-scale progressive fusion) is effective but expensive: the decoder alone uses
50–100M parameters depending on backbone size. Two recent lines of work address this.

### AnyDepth / Simple Depth Transformer (SDT) — Ren et al., 2026

SDT inverts DPT's order to **fuse-then-reassemble**: all ViT layer tokens are first
fused with a lightweight cls-token readout mechanism, then undergo a single-path
upsampling pass:

$$\text{DPT: } \underbrace{\text{Readout}_l \to \text{Reassemble}_l}_{\text{per layer}} \to \text{Fuse across scales}$$

$$\text{SDT: } \underbrace{\text{Fuse all layers}}_{\text{once}} \to \text{Reassemble} \to \text{Upsample}$$

| Backbone | DPT params | SDT params | Reduction |
|----------|-----------|------------|----------|
| ViT-S | 50.8M | 5.5M | **89%** |
| ViT-B | 76.1M | 9.5M | **88%** |
| ViT-L | 99.6M | 13.4M | **87%** |

Combined with DINOv3 encoding and quality-based data filtering, SDT matches or
exceeds DPT accuracy on five benchmarks while reducing compute and memory by an
order of magnitude — a strong signal that depth decoders were over-parameterised.

### FoundationGeo — Learning Pixel-Wise Spatial Fields (2026)

Takes a different direction: instead of predicting affine-ambiguous depth, FoundationGeo
predicts **three** pixel-wise fields simultaneously:
- Metric depth $d(u,v)$
- Camera ray direction $\mathbf{r}(u,v)$
- Spatial scale calibration field

By embedding per-pixel intrinsics information, FoundationGeo achieves the best average
rank across seven benchmarks (indoor + outdoor + challenging datasets) with **no**
camera-specific fine-tuning — directly addressing the scale ambiguity problem that
requires post-hoc alignment in standard pipelines.

### Marigold V2 — Diffusion Transformers for Depth (2026)

Converts an image-editing DiT into a depth regressor via:
1. **iREPA-depth**: representation alignment against semantic features from
   ground-truth depth (not RGB)
2. **SinkLoss**: Sinkhorn matching objective for edge-sharp predictions
3. **QLoRA fine-tuning**: 4-bit quantisation makes training possible on a single
   consumer GPU

Trade-off: highest visual quality and sharpest edges, but **generative** (slower,
non-deterministic). For real-time SLAM, discriminative models (SDT, FoundationGeo)
remain preferred; for offline reconstruction, Marigold V2 produces the crispest results.

### Practical Guidance

| Scenario | Recommended model | Why |
|----------|------------------|-----|
| Real-time drone SLAM | SDT + DA3 encoder | Fast, light, accurate |
| Metric depth (no calib.) | FoundationGeo | Intrinsics-free metric |
| Offline high-quality recon. | Marigold V2 | Sharpest edges |
| Resource-constrained | SDT + ViT-S | 5.5M decoder params |

---
## Summary

### Key Equations

| Concept | Formula |
|:---|:---|
| Affine ambiguity | $d_{\text{metric}} = \alpha \cdot d_{\text{net}} + \beta$ |
| Attention | $\text{Attn}(Q,K,V) = \text{softmax}\!\left(\dfrac{QK^T}{\sqrt{d_k}}\right)V$ |
| Multi-head | $\text{MHSA} = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W_O$ |
| abs_rel | $\dfrac{1}{N}\sum \dfrac{|\hat{d}_i - d_i^*|}{d_i^*}$ |
| $\delta_k$ | $\text{frac where } \max(\hat{d}/d^*, d^*/\hat{d}) < 1.25^k$ |
| SI loss | $\dfrac{1}{N}\sum e_i^2 - \dfrac{\lambda}{N^2}(\sum e_i)^2$ |
| Grad loss | $\dfrac{1}{N}\sum (|\partial_x e| + |\partial_y e|)$ |
| SDT complexity | $\mathcal{O}(\text{fuse once}) \ll \mathcal{O}(\text{per-layer reassemble})$ |

### PromptDepthAnything++ — Promptable Metric Depth (2025)

A new paradigm: instead of training separate metric models per domain, **prompt** a
foundation model with sparse metric measurements (e.g., 32–128 LiDAR points):

$$d_\text{metric}(u,v) = f_\theta(I, \{(u_i, v_i, d_i^{\text{LiDAR}})\}_{i=1}^N)$$

Key design choices:
- **Pattern-agnostic prompting**: works with any sparse depth pattern (LiDAR, radar, SfM points)
- **4K resolution output** (3840×2160) — highest among current depth models
- The prompt acts as a "metric anchor" that resolves the affine ambiguity without domain-specific fine-tuning

This is directly relevant for drone SLAM: a downward-facing LiDAR or barometer provides
sparse metric depth that can prompt a monocular depth model, yielding dense metric depth
maps without the usual scale/shift alignment step.

### Key Takeaways

1. Monocular depth is ill-posed — networks learn strong geometric priors
2. Output is affine-ambiguous — align with scale α and shift β
3. DPT uses ViT tokens → multi-scale reassembly → progressive fusion
4. Self-attention lets every token attend to every other (global context)
5. Depth Anything v2/v3 + SDT decoder achieves DPT-level accuracy at 10× fewer params
6. DUSt3R/MASt3R predict 3D pointmaps directly, bypassing classical pipelines
7. FoundationGeo resolves scale ambiguity without camera-specific calibration
8. **Promptable depth** (PromptDepthAnything++) fuses sparse metric sensors with foundation models
9. Seven standard metrics capture complementary aspects of depth quality

### What's Next

→ **Notebook 10**: Structure from Motion — recovering 3D from multiple unordered images